# RandomForestRegressor

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from mpl_toolkits.mplot3d import Axes3D

# Assuming 'your_file.csv' is the name of your CSV file
df = pd.read_csv('dataset_10.csv')

# Selecting relevant features and target variables
features = ['RC.aileron', 'RC.elevator', 'RC.throttle', 'RC.rudder', 'RC.mode']
position_targets = ['OSD.longitude', 'OSD.latitude', 'OSD.height [ft]']
orientation_targets = ['OSD.pitch', 'OSD.roll', 'OSD.yaw']

# Combining features for both position and orientation
all_features = features + orientation_targets

# Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df[all_features], df[position_targets + orientation_targets], test_size=0.2, random_state=42
)

# Handling missing values in input features using SimpleImputer
imputer = SimpleImputer(strategy='mean')  
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Handling missing values in target variables (if any)
y_train = y_train.dropna().reset_index(drop=True)
X_train_imputed = X_train_imputed[y_train.index]

# Creating and training the Random Forest Regression models for position
models_position = {}
for target_variable in position_targets:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_imputed, y_train[target_variable])
    models_position[target_variable] = model

# Creating and training the Random Forest Regression models for orientation
models_orientation = {}
for target_variable in orientation_targets:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train_imputed, y_train[target_variable])
    models_orientation[target_variable] = model

# Making predictions on the test set for position
position_pred = {}
for target_variable, model in models_position.items():
    position_pred[target_variable] = model.predict(X_test_imputed)

# Making predictions on the test set for orientation
orientation_pred = {}
for target_variable, model in models_orientation.items():
    orientation_pred[target_variable] = model.predict(X_test_imputed)

# Create a 3D plot for the predicted drone path
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the actual drone path
ax.scatter(y_test['OSD.longitude'], y_test['OSD.latitude'], y_test['OSD.height [ft]'], c='blue', marker='o', label='Actual Path')

# Plot the predicted drone path
ax.scatter(position_pred['OSD.longitude'], position_pred['OSD.latitude'], position_pred['OSD.height [ft]'], c='red', marker='x', label='Predicted Path')

# Customize the plot
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_zlabel('Altitude (ft)')
ax.set_title('Actual vs. Predicted Drone Path')
ax.legend()

# Show the plot
plt.show()

# Plotting the orientation (pitch, roll, yaw) separately
fig_orientation, ax_orientation = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Plot the actual orientation
for i, target_variable in enumerate(orientation_targets):
    ax_orientation[i].plot(y_test.index, y_test[target_variable], label=f'Actual {target_variable}', marker='o')

# Plot the predicted orientation
for i, target_variable in enumerate(orientation_targets):
    ax_orientation[i].plot(y_test.index, orientation_pred[target_variable], label=f'Predicted {target_variable}', marker='x')

# Customize the orientation plots
for i, target_variable in enumerate(orientation_targets):
    ax_orientation[i].set_ylabel(f'{target_variable}')
    ax_orientation[i].legend()

# Show the orientation plots
plt.xlabel('Sample Index')
plt.suptitle('Actual vs. Predicted Drone Orientation')
plt.show()
